# FairHealth — Example Notebook
**Trustworthy Healthcare AI**

This notebook demonstrates FairHealth's three core modules:
- `fairhealth.fairness` — demographic parity audit
- `fairhealth.explain` — fuzzy + SHAP explainability  
- `fairhealth.federated` — privacy-preserving federated learning

📄 Papers: MobiHealth 2026 · ICAIHE 2026 · CIBB 2026 · CCAI 2026
🔗 GitHub: https://github.com/Farjana-Yesmin/fairhealth

In [ ]:
!pip install fairhealth ucimlrepo -q
print('✓ FairHealth installed')

## 1. Load Maternal Health Data (Bangladesh Context)

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np

maternal   = fetch_ucirepo(id=863)
patient_df = maternal.data.original
print(f'Patients: {len(patient_df)}')
print(f'Risk levels: {patient_df["RiskLevel"].value_counts().to_dict()}')
patient_df.head()

## 2. Fairness Audit (from MobiHealth 2026 paper)

In [ ]:
from fairhealth.fairness.metrics import demographic_parity_diff, fairness_report
import numpy as np

# Age groups as sensitive attribute
patient_df['AgeGroup'] = pd.cut(
    patient_df['Age'], bins=[0,25,35,45,100],
    labels=['Teen(<=25)','Adult(26-35)','Middle(36-45)','Older(46+)']
)

# Simple risk prediction based on blood pressure
y_pred    = (patient_df['SystolicBP'] >= 130).astype(int).values
sensitive = patient_df['AgeGroup'].astype(str).values

dpd = demographic_parity_diff(y_pred, sensitive)
print(f'Demographic Parity Difference: {dpd:.4f}')
fairness_report(y_pred, (patient_df['RiskLevel']=='high risk').astype(int).values,
                sensitive, 'AgeGroup')

## 3. Fuzzy-XGBoost Explanation (from ICAIHE 2026 paper)

In [ ]:
from fairhealth.explain.fuzzy import get_fired_rules, score_to_label

# Test with a high-risk patient profile
print('=== HIGH-RISK PATIENT ===')
print('Age:42, SBP:145, BS:12.0, HR:88')
rules = get_fired_rules(age=42, sbp=145, bs=12.0, hr=88)
print(f'Rules fired: {len(rules)}')
for r in rules:
    print(f'  Rule {r["id"]}: {r["condition"]} -> {r["outcome"]}')

print('\n=== LOW-RISK PATIENT ===')
print('Age:22, SBP:95, BS:6.5, HR:70')
rules2 = get_fired_rules(age=22, sbp=95, bs=6.5, hr=70)
for r in rules2:
    print(f'  Rule {r["id"]}: {r["condition"]} -> {r["outcome"]}')

## 4. Federated Privacy (from MedHE CIBB 2026 paper)

In [ ]:
from fairhealth.federated.privacy import sparsify, add_gaussian_noise, clip_weights
import numpy as np

# Simulate model weights
np.random.seed(42)
weights = np.random.randn(100)  # 100-dim weight vector

# Apply gradient sparsification
sparse_w, rate = sparsify(weights, sparsity=0.975)
print(f'Sparsification: {rate:.1%} of weights zeroed')
print(f'Communication reduced: {rate:.1%}')

# Apply differential privacy
clipped  = clip_weights(weights, clip_norm=1.0)
dp_w     = add_gaussian_noise(clipped, epsilon=1.0)
print(f'DP noise added: epsilon=1.0 (strong privacy)')
print(f'Noise magnitude: {np.abs(dp_w - clipped).mean():.4f}')
print(f'\nMedHE paper result: 97.5% comm reduction, macro-F1=0.950')